## Linear Regression

In [1]:
import numpy as np
import random
import torch
import torch.nn as nn 

In [2]:
# weight w has shape (p, 1)
# bias b is a scalar
# data x has shape (p, 1)
p = 10 # number of data dimension
x = np.zeros((p, 1))
w = np.zeros((p, 1))
b = 0
y_hat = (x * w).sum() + b

In [3]:
#`features` shape is (n, p), `labels` shape is (n, 1)
# n is number of sample
num_epochs = 100 # epoch size
batch_size = 3 # mini batch
features = torch.normal(0, 0.01, size=(num_epochs, p))
labels = torch.normal(0, 0.01, size=(num_epochs, 1))
learning_rate = 0.01


def data_iter(batch_size, features, labels):
  num_examples = len(features)
  indices = list(range(num_examples))
  random.shuffle(indices) # read example at random
  for i in range(0, num_examples, batch_size):
    batch_indices = torch.tensor(
      indices[i:min(i + batch_size, num_examples)]
    )
    yield features[batch_indices], labels[batch_indices]
    
w = torch.normal(0, 0.01, size=(p, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)


for epoch in range(num_epochs):
  for X, y in data_iter(batch_size, features, labels):
    y_hat = X @ w + b
    loss = ((y_hat - y)**2 / 2).mean()
    loss.backward()
    with torch.no_grad():
      for param in [w, b]:
        param -= learning_rate * param.grad
        param.grad.zero_()

## MLP

### Perceptron

In [4]:
num_inputs = 10
num_hiddens = 5
num_outputs = 3

In [14]:
def relu(X):
  return torch.max(X, 0)[0]

W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens))
W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs))

H = relu(X @ W1 + b1)
H.size()
Y = H @ W2 + b2

### Convolutionary Layer

In [25]:
# both input `X` and weight `K` are matrices
K = torch.randn(1, 1)

print(X.shape)
h, w = K.shape
Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
stride = 1
for i in range(Y.shape[0]):
  for j in range(Y.shape[1]):
    Y[i, j] = (X[i : i + h, j : j + w] * K).sum()
print(Y.shape)

torch.Size([1, 10])
torch.Size([1, 10])


### Pooling Layer

In [26]:
# h, w: pooling window height and width
# mode: max or avg

mode = 'max'

Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))

for i in range(Y.shape[0]):
  for j in range(Y.shape[1]):
    if mode == 'max':
      Y[i, j] = (X[i : i + h, j : j + w] * K).max()
    elif mode == 'avg':
      Y[i, j] = (X[i : i + h, j : j + w] * K).mean()
print(Y.shape)

torch.Size([1, 10])


### RNN

In [27]:
W_xh = nn.Parameter(torch.randn(num_inputs, num_hiddens) * 0.01)
W_hh = nn.Parameter(torch.randn(num_hiddens, num_hiddens) * 0.01)
b_h = nn.Parameter(torch.zeros(num_hiddens))

H = torch.zeros(num_hiddens)
outputs = []

inputs = torch.randn(10, batch_size, num_inputs)
for X in inputs: # `inputs` shape: (num_steps, batch_size, num_inputs)
  H = torch.tanh(X @ W_xh + H @ W_hh + b_h)
  outputs.append(H)